# Load an Existing Chroma Database

This notebook reconnects to the persisted Chroma collection created in the CRUD notebook and reads its contents.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

## 1. Rebuild the Same Configuration

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()

# Walk up the directory tree until the folder name matches your target root
while project_root.name != "adv-rag-mastery" and project_root != project_root.parent:
    project_root = project_root.parent

project_root

PosixPath('/Users/lokeshkv/data-engineering/learning/adv-rag-mastery')

In [3]:
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: /Users/lokeshkv/data-engineering/learning/adv-rag-mastery/.env


In [4]:
vector_store_path = Path.cwd()
if vector_store_path.name == "notebooks":
    vector_store_path = vector_store_path.parent

vector_store_path

PosixPath('/Users/lokeshkv/data-engineering/learning/adv-rag-mastery/04_vector_stores')

In [5]:
collection_name = "demo_2"
persist_directory = vector_store_path / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: /Users/lokeshkv/data-engineering/learning/adv-rag-mastery/04_vector_stores/notebooks/chroma_langchain_db


In [6]:
# Use the same embedding model that was used to build the store.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Connected to the existing Chroma collection.")

Connected to the existing Chroma collection.


## 2. Add Small Display Helpers

In [7]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_stored_documents(records):
    """Print stored Chroma records in a readable format."""
    ids = records.get("ids", [])
    documents = records.get("documents", [])
    metadatas = records.get("metadatas", [])

    print(f"Total documents in collection: {len(ids)}")
    print()

    for index, (doc_id, document_text, metadata) in enumerate(zip(ids, documents, metadatas), start=1):
        print(f"{index}. id={doc_id}")
        print(f"   topic={metadata.get('topic')} | doc_number={metadata.get('doc_number')}")
        print(f"   content={preview_text(document_text)}")
        print()

## 3. Fetch and Inspect the Stored Records

In [8]:
stored_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
stored_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [9]:
stored_records["embeddings"].shape

(0,)

In [10]:
print_stored_documents(stored_records)

Total documents in collection: 0

